Groundwater | Transport Modeling Track

# Step 4: Model Implementation — Solute Transport and the Grid–Péclet Problem (MODFLOW 6 GWT)

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → **4-Build** → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In Steps 1–3 you set objectives for a conservative tracer in the Limmat Valley aquifer ([01t](01t_model_goal.ipynb)), built a perceptual model of a **groundwater heat-exchange (GWHE) doublet** — paired injection/abstraction wells, studied here as a solute-transport problem ([02t](02t_perceptual_model.ipynb)), and translated it into the MODFLOW 6 **GWT** packages and the numerical-accuracy criteria ([03t](03t_modflow_transport.ipynb)). Now we **build the coupled flow-and-transport model** and use it to confront the single most consequential numerical choice in transport modelling: **grid resolution**.

| | |
|---|---|
| **Time** | ~60–75 min |

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Build** a coupled steady GWF6 / transient GWT6 doublet on the inherited flow grid, with the source set by an **injection well + SSM auxiliary concentration**
2. **Diagnose** a grid with the longitudinal and transverse **grid Péclet numbers**, and explain why $Pe \le 2$ and $Cr \le 1$ are *accuracy* criteria, **not** stability limits
3. **Quantify** how a coarse grid **under-predicts the peak concentration** along the source→receptor corridor, and how local **refinement** recovers most of it — and likewise recovers the flux-integrated recirculation fraction the abstraction well sees (a quantity that *converges* under refinement), unlike the transverse spreading
4. **Judge** which threshold claims are defensible (longitudinal / flux-integrated quantities the grid converges on) and which are numerical artefacts that no feasible grid can fix

> ### This is a BUILD/DEMO notebook — you MODIFY, you do not build from scratch
> The doublet location is **pinned to a real licensed concession** (`b010191`, a spare pair not assigned to any student group). You will **modify** parameters (injection rate, concentration) and **read** diagnostics off the model — you do **not** free-pick the well location or design the refinement. This keeps every student looking at the same, numerically well-behaved demonstration.

## Prerequisites
- **Flow track** — especially [04f_model_implementation.ipynb](../flow/04f_model_implementation.ipynb)
- **Transport Steps 1–3** — [01t](01t_model_goal.ipynb), [02t](02t_perceptual_model.ipynb), [03t](03t_modflow_transport.ipynb)

> **Where does data go?** Inputs come from `~/applied_groundwater_modelling_data/`; the two doublet runs are cached under `transport_doublet_model/`.

## Introduction — the question, and the trap

A GWHE doublet injects water at one well and abstracts it at a second well a short distance away. The operational question is blunt: **does what we inject short-circuit back to the abstraction well, and at what concentration?** That recirculation decides whether the scheme works (thermal short-circuiting) or whether a contaminant reaches the receptor.

Here is the trap. The plume between the two wells is **narrow** — only as wide as transverse dispersion lets it spread ($\alpha_T = 1$ m). The inherited flow grid has ~50 m cells. When a model cell is far larger than the spreading length, the numerical scheme adds **numerical dispersion**: it smears the plume — most damagingly **sideways**, across the flow — and dilutes the concentration along the centre-line that the abstraction well draws from.

The flagship lesson of this notebook: **a coarse grid makes you UNDER-ESTIMATE the peak concentration in the source→receptor corridor.** We will measure that under-estimate, recover most of it by refining the corridor, and then show the part that refinement *cannot* fix — so you know which numbers to trust. (The same coarse-grid dilution also under-reads the *flux-integrated* recirculation fraction at the well — but, unlike the lateral plume width, that fraction **converges** as you refine, which is exactly why it, and not a contour or a width, is the number you can defend.)

---
## 1 — Setup

In [ ]:
import sys, os, time, shutil, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString

import flopy
from flopy.mf6 import MFSimulation

sys.path.insert(0, '../../_SUPPORT/src')
sys.path.insert(0, '../../_SUPPORT/src/scripts/scripts_exercises')

from data_utils import download_named_file, get_default_data_folder
from model_io_utils import load_base_simulation, ensure_flow_model

# Validated GWT-solute base-model module (M2). We reuse its locked parameters,
# its corridor-refine builder, and its Courant-sizing helper so the build is
# correct and the cells stay readable.
import transport_base_model as tbm
from transport_base_model import (LOCKED_PARAMS, build_doublet_base,
                                  load_doublet_base, courant_nstp, _cellsize)

from shared_functions import check_task_with_solution, create_multiple_choice

print('FloPy', flopy.__version__)
print('Locked transport parameters:')
for k in ['alh', 'ath1', 'diffc', 'porosity', 'scheme', 'base_cell_size', 'refined_cell_size']:
    print(f'   {k:18s} = {LOCKED_PARAMS[k]}')

In [ ]:
DATA_DIR   = get_default_data_folder()
FLOW_WS    = str(ensure_flow_model())                          # 05f-CALIBRATED steady flow model (mean K ~361 m/d)
WORK_WS    = os.path.join(DATA_DIR, 'transport_doublet_model')  # this notebook's outputs
NATIVE_WS  = os.path.join(WORK_WS, 'native')                    # doublet on the ~50 m grid
REFINED_WS = os.path.join(WORK_WS, 'refined')                   # doublet on the 10 m corridor

# ── Execution control ──────────────────────────────────────────────────────
# FORCE_RERUN = False  → re-load the cached native + refined runs (a normal pass; seconds)
# FORCE_RERUN = True   → rebuild and re-solve both doublets (~70–90 s cold on a laptop)
FORCE_RERUN = False

os.makedirs(WORK_WS, exist_ok=True)
print('Flow model:', FLOW_WS)
print('Workspace :', WORK_WS)
print('FORCE_RERUN =', FORCE_RERUN)

---
## 2 — Load the flow model

We load the **05f-calibrated** steady-state Limmat flow model (via `ensure_flow_model()`): a single-layer DISV grid (~50 m cells) with the *calibrated* $K$ field (mean $K \approx 361$ m/d), river (RIV) and constant-head (CHD) boundaries, and recharge. The transport model will run **on this grid** (native) and on a **locally refined copy** of it (refined) — the flow solution is the velocity field that advects the solute.

In [ ]:
sim_flow = load_base_simulation(FLOW_WS)
gwf_flow = sim_flow.get_model()
mf6_exe  = gwf_flow.simulation.exe_name

modelgrid = gwf_flow.modelgrid
xc = np.array(modelgrid.xcellcenters)
yc = np.array(modelgrid.ycellcenters)
ncpl_native = len(xc)
k_native = gwf_flow.npf.k.array

# Domain polygon + Limmat/Sihl rivers — needed only for the refined-grid rebuild (§6)
boundary_gdf = gpd.read_file(download_named_file(name='model_boundary', data_type='gis'))
rivers_gdf   = gpd.read_file(download_named_file(name='rivers', data_type='gis'))
rivers_gdf   = rivers_gdf[rivers_gdf['GEWAESSERNAME'].isin(['Limmat', 'Sihl'])
                          & rivers_gdf.intersects(boundary_gdf.geometry.iloc[0])]

print(f'Inherited grid : {ncpl_native} DISV cells, 1 layer')
print(f'K range        : {k_native.min():.0f} – {k_native.max():.0f} m/d')
print(f'Median cell size: {np.median(_cellsize(modelgrid, ncpl_native)):.0f} m')

---
## 3 — Pin the doublet and the locked transport parameters

We place the **GWHE doublet** (groundwater heat-exchange: paired injection/abstraction wells, treated here as a conservative-solute problem) at the coordinates of a **real licensed concession, `b010191`** — a spare pair not assigned to any student group. Its wells are resolved from the GIS **by role** (FASSART): the injection (reinjection / *Rückgabe*) well and the abstraction (*Entnahme*) well are **200 m apart**, oriented broadly downgradient in the regional Limmat flow field. By design this pair is partially recirculating — some injectate short-circuits to the abstraction well, some escapes downgradient — which is exactly the risk we want to quantify, at a realistic concession yield ($Q = 5760$ m³/d, the $>3000$ l/min class).

The transport parameters are **locked** for the whole M2/M3/M4 track (`LOCKED_PARAMS`): $\alpha_L = 10$ m, $\alpha_T = 1$ m, $n_e = 0.20$, molecular diffusion $D_m = 8.64\times10^{-5}$ m²/d, **TVD** advection. The source is set by an **injection well carrying an auxiliary concentration**, coupled to transport through SSM (mass loading $= Q_{inj}\,c_{inj}$) — *not* a constant-concentration (CNC) cell, because a real doublet injects water, not a fixed concentration.

> **You may MODIFY** `Q_INJ` and `C_INJ` below and re-run with `FORCE_RERUN = True`. **Do not** move `INJ_XY` / `ABS_XY` or change the refinement — those are pinned to the real concession.

In [ ]:
# Real concession doublet b010191 (LV95) — a licensed GWHE well pair, spare (not assigned to any student group)
# Wells resolved from the gwhe_wells GIS by role (FASSART): injection = Rückgabe/Rückversickerung/Anreicherung,
# extraction = Entnahme/Ertrag. Here: injection = b010191_05 (Rückgabe), extraction = b010191_01 (Entnahme).
INJ_XY     = (2681297.0, 1248917.0)   # injection well   (role: Rückgabe / reinjection)   (PINNED)
ABS_XY     = (2681487.0, 1248981.0)   # abstraction well (role: Entnahme / extraction)     (PINNED)
Q_INJ      = 5760.0                    # injection / abstraction rate [m^3/d] (>3000 l/min yield class)  (MODIFY-able)
C_INJ      = 1.0                       # injected relative concentration [-]    (MODIFY-able)
TOTAL_TIME = 365.0                     # simulation length [d]

# The two wells are the real licensed coordinates — the spacing is fixed by the concession, not by us.
spacing = float(np.hypot(ABS_XY[0] - INJ_XY[0], ABS_XY[1] - INJ_XY[1]))

print(f'Injection well   : {INJ_XY}')
print(f'Abstraction well : {ABS_XY}  — {spacing:.0f} m from the injection well')
print(f'Q = +/-{Q_INJ:.0f} m3/d   c_inj = {C_INJ}   t = {TOTAL_TIME:.0f} d')

---
## 4 — Build the native GWT doublet

We build **one coupled simulation**: a steady-state GWF6 flow model (Newton, COMPLEX/BICGSTAB IMS, storage steady for all periods) and a transient GWT6 transport model (TVD advection), linked by `GWF6-GWT6`. The injection well carries an `auxiliary='CONCENTRATION'` column; the GWT **SSM** package reads it as the mass source. NPF saves both `specific_discharge` and `flows` so we can recover the seepage velocity. The number of time steps is sized from the peak near-well velocity so that $Cr \approx 0.9$ (per-cell, with a sliver floor — the `courant_nstp` helper).

> **Stability vs accuracy — read this once.** MF6 solves GWT advection **implicitly** (through IMS), and TVD is a higher-order, dispersion-limiting scheme. The model stays **stable** far above $Pe = 2$ and $Cr = 1$ — it does **not** blow up. What degrades above those thresholds is **accuracy**: numerical dispersion (space) and breakthrough smearing (time). Keep that distinction in mind for the rest of the notebook.

In [ ]:
def _build_native_sim(sim_ws, nstp):
    # Coupled GWF(steady,NEWTON)-GWT(transient,TVD) doublet on the INHERITED ~50 m grid.
    # Mirrors transport_base_model._make_sim, but with NO corridor refinement.
    LP = LOCKED_PARAMS
    disv = gwf_flow.get_package('disv')
    gp = dict(nvert=disv.nvert.get_data(), vertices=disv.vertices.get_data(),
              cell2d=disv.cell2d.get_data())
    top, botm = disv.top.array, disv.botm.array
    heads = gwf_flow.output.head().get_data().flatten()
    chd = gwf_flow.get_package('chd').stress_period_data.get_data(0)
    riv = gwf_flow.get_package('riv').stress_period_data.get_data(0)
    rch = gwf_flow.get_package('rcha').recharge.get_data()
    inj_c = int(np.argmin((xc - INJ_XY[0])**2 + (yc - INJ_XY[1])**2))
    abs_c = int(np.argmin((xc - ABS_XY[0])**2 + (yc - ABS_XY[1])**2))

    sim = MFSimulation(sim_name='nat', exe_name=mf6_exe, sim_ws=sim_ws)
    flopy.mf6.ModflowTdis(sim, time_units=LP['time_units'], nper=1,
                          perioddata=[(TOTAL_TIME, nstp, 1.0)])
    flopy.mf6.ModflowIms(sim, filename='gwf.ims', **tbm._GWF_IMS)
    gwf = flopy.mf6.ModflowGwf(sim, modelname='gwf', save_flows=True,
                               newtonoptions=tbm._GWF_NEWTON)
    flopy.mf6.ModflowGwfdisv(gwf, nlay=1, ncpl=len(xc), nvert=gp['nvert'], top=top,
                             botm=botm, vertices=gp['vertices'], cell2d=gp['cell2d'])
    flopy.mf6.ModflowGwfnpf(gwf, icelltype=1, k=k_native,
                            save_flows=True, save_specific_discharge=True)
    flopy.mf6.ModflowGwfic(gwf, strt=np.maximum(heads, botm[0] + 0.01))
    flopy.mf6.ModflowGwfsto(gwf, steady_state={0: True})
    flopy.mf6.ModflowGwfrcha(gwf, recharge=rch)
    flopy.mf6.ModflowGwfchd(gwf, stress_period_data=[(tuple(r['cellid']), float(r['head']))
                                                     for r in chd])
    flopy.mf6.ModflowGwfriv(gwf, stress_period_data=[(tuple(r['cellid']), float(r['stage']),
                            float(r['cond']), float(r['rbot'])) for r in riv])
    # source = injection WEL + AUX concentration ; paired abstraction WEL
    flopy.mf6.ModflowGwfwel(gwf, pname='injw', auxiliary=['CONCENTRATION'],
                            stress_period_data={0: [[(0, inj_c), Q_INJ, C_INJ]]})
    flopy.mf6.ModflowGwfwel(gwf, pname='absw',
                            stress_period_data={0: [[(0, abs_c), -abs(Q_INJ)]]})
    flopy.mf6.ModflowGwfoc(gwf, head_filerecord='gwf.hds', budget_filerecord='gwf.cbc',
                           saverecord=[('HEAD', 'LAST'), ('BUDGET', 'LAST')])
    gwt = flopy.mf6.ModflowGwt(sim, modelname='gwt', save_flows=True)
    flopy.mf6.ModflowGwtdisv(gwt, nlay=1, ncpl=len(xc), nvert=gp['nvert'], top=top,
                             botm=botm, vertices=gp['vertices'], cell2d=gp['cell2d'])
    flopy.mf6.ModflowGwtic(gwt, strt=0.0)
    flopy.mf6.ModflowGwtmst(gwt, porosity=LP['porosity'])
    flopy.mf6.ModflowGwtadv(gwt, scheme=LP['scheme'])
    flopy.mf6.ModflowGwtdsp(gwt, alh=LP['alh'], ath1=LP['ath1'],
                            diffc=LP['diffc'], xt3d_off=LP['xt3d_off'])
    flopy.mf6.ModflowGwtssm(gwt, sources=[['injw', 'AUX', 'CONCENTRATION']])
    flopy.mf6.ModflowGwtoc(gwt, concentration_filerecord='gwt.ucn',
                           saverecord=[('CONCENTRATION', 'ALL')])
    flopy.mf6.ModflowIms(sim, filename='gwt.ims', **tbm._GWT_IMS)
    sim.register_ims_package(sim.get_package('gwt.ims'), ['gwt'])
    flopy.mf6.ModflowGwfgwt(sim, exgtype='GWF6-GWT6', exgmnamea='gwf', exgmnameb='gwt',
                            filename='nat.gwfgwt')
    sim.write_simulation(silent=True)
    ok, buf = sim.run_simulation(silent=True)
    if not ok:
        raise RuntimeError('native run failed:\n' + '\n'.join(buf[-12:]))
    return sim


def build_or_load_native(ws, force):
    sim_ws = os.path.join(ws, 'sim')
    if (not force) and os.path.exists(os.path.join(sim_ws, 'gwt.ucn')):
        return MFSimulation.load(sim_ws=sim_ws, exe_name=mf6_exe, verbosity_level=0), 'cached'
    if os.path.exists(ws):
        shutil.rmtree(ws)
    os.makedirs(sim_ws)
    # pilot run → read velocity field → size Courant → production run
    sim = _build_native_sim(sim_ws, nstp=20)
    gwf = sim.get_model('gwf')
    csz = _cellsize(gwf.modelgrid, len(xc))
    spd = gwf.output.budget().get_data(text='DATA-SPDIS')[0]
    vmag = np.sqrt(spd['qx']**2 + spd['qy']**2) / LOCKED_PARAMS['porosity']
    line = LineString([INJ_XY, ABS_XY])
    corr = np.array([line.distance(Point(xc[i], yc[i])) < LOCKED_PARAMS['refine_radius']
                     for i in range(len(xc))])
    ic = int(np.argmin((xc - INJ_XY[0])**2 + (yc - INJ_XY[1])**2))
    ac = int(np.argmin((xc - ABS_XY[0])**2 + (yc - ABS_XY[1])**2))
    cnw = corr.copy(); cnw[[ic, ac]] = False
    nstp, _, _, _ = courant_nstp(vmag, csz, cnw, TOTAL_TIME)
    return _build_native_sim(sim_ws, nstp=nstp), 'built'


t0 = time.time()
sim_nat, status_nat = build_or_load_native(NATIVE_WS, FORCE_RERUN)
gwf_nat, gwt_nat = sim_nat.get_model('gwf'), sim_nat.get_model('gwt')
print(f'Native doublet [{status_nat}] in {time.time() - t0:.0f} s')

In [ ]:
def doublet_diagnostics(gwf, gwt, src_xy, abs_xy):
    # Pull Peclet, Courant, breakthrough and the final plume field off a solved doublet.
    mg = gwf.modelgrid
    xcc, ycc = np.array(mg.xcellcenters), np.array(mg.ycellcenters); n = len(xcc)
    csz = _cellsize(mg, n)
    aL, aT = LOCKED_PARAMS['alh'], LOCKED_PARAMS['ath1']
    src_c = int(np.argmin((xcc - src_xy[0])**2 + (ycc - src_xy[1])**2))
    abs_c = int(np.argmin((xcc - abs_xy[0])**2 + (ycc - abs_xy[1])**2))
    mid = ((src_xy[0] + abs_xy[0]) / 2, (src_xy[1] + abs_xy[1]) / 2)
    mid_c = int(np.argmin((xcc - mid[0])**2 + (ycc - mid[1])**2))
    line = LineString([src_xy, abs_xy])
    corr = np.array([line.distance(Point(xcc[i], ycc[i])) < LOCKED_PARAMS['refine_radius']
                     for i in range(n)])
    cobj = gwt.output.concentration(); times = np.array(cobj.get_times())
    bt_well = np.maximum([cobj.get_data(totim=t)[0, 0, abs_c] for t in times], 0)
    bt_mid  = np.maximum([cobj.get_data(totim=t)[0, 0, mid_c] for t in times], 0)
    field   = np.maximum(cobj.get_data(totim=times[-1])[0, 0], 0)
    spd = gwf.output.budget().get_data(text='DATA-SPDIS')[0]
    vmag = np.sqrt(spd['qx']**2 + spd['qy']**2) / LOCKED_PARAMS['porosity']
    # binding Courant cell: largest v/Δs in the corridor, excluding wells and sub-resolution slivers
    floor = 0.4 * LOCKED_PARAMS['refined_cell_size']
    sel = corr.copy(); sel[[src_c, abs_c]] = False; sel &= (csz >= floor)
    j = int(np.where(sel)[0][np.argmax(vmag[sel] / csz[sel])])
    return dict(mg=mg, xcc=xcc, ycc=ycc, csz=csz, src_c=src_c, abs_c=abs_c, corr=corr,
                times=times, bt_well=bt_well, bt_mid=bt_mid, field=field,
                dx_src=float(csz[src_c]), PeL=float(csz[src_c] / aL), PeT=float(csz[src_c] / aT),
                recirc=float(np.median(bt_well[-8:])), mid_peak=float(bt_mid.max()),
                field_max=float(field.max()),
                v_bind=float(vmag[j]), ds_bind=float(csz[j]), dt_max=float(csz[j] / vmag[j]))


dn = doublet_diagnostics(gwf_nat, gwt_nat, INJ_XY, ABS_XY)
print(f"NATIVE doublet : {len(dn['xcc'])} cells, source-cell size dx = {dn['dx_src']:.0f} m")
print(f"  grid Peclet : Pe_L = {dn['PeL']:.1f}   Pe_T = {dn['PeT']:.0f}")
print(f"  recirculation to abstraction well : {dn['recirc']:.3f} of c_inj")
print(f"  peak concentration mid-corridor   : {dn['mid_peak']:.3f} of c_inj")

In [ ]:
def plot_plume(ax, d, title, zoom=200):
    pmv = flopy.plot.PlotMapView(modelgrid=d['mg'], ax=ax)
    ca = pmv.plot_array(d['field'], cmap='magma_r', vmin=0, vmax=1)
    pmv.plot_grid(lw=0.15, color='0.6', alpha=0.35)
    ax.plot(*INJ_XY, 'o', mfc='lime', mec='k', ms=9)
    ax.plot(ABS_XY[0], ABS_XY[1], 's', mfc='deepskyblue', mec='k', ms=9)
    ax.set_xlim(INJ_XY[0] - zoom, INJ_XY[0] + zoom)
    ax.set_ylim(INJ_XY[1] - zoom, INJ_XY[1] + zoom)
    ax.set_title(title); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    return ca

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.2))
ca = plot_plume(ax1, dn, f"Native plume  (Pe_L={dn['PeL']:.1f}, Pe_T={dn['PeT']:.0f})")
fig.colorbar(ca, ax=ax1, shrink=0.8, label='C / c_inj')
ax1.plot([], [], 'o', mfc='lime', mec='k', label='injection')
ax1.plot([], [], 's', mfc='deepskyblue', mec='k', label='abstraction')
ax1.legend(loc='upper right', fontsize=8)

ax2.plot(dn['times'], dn['bt_well'], color='firebrick', lw=2, label='at abstraction well')
ax2.plot(dn['times'], dn['bt_mid'],  color='goldenrod', lw=1.5, ls='--', label='mid-corridor')
ax2.set_xlabel('time (d)'); ax2.set_ylabel('C / c_inj'); ax2.set_ylim(0, 1)
ax2.set_title('Native breakthrough'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 5 — Grid–Péclet diagnosis of the native grid

The **grid Péclet number** compares the cell size to the dispersive spreading length:

$$Pe_L = \frac{\Delta x}{\alpha_L}, \qquad Pe_T = \frac{\Delta x}{\alpha_T}$$

The accuracy guideline is $Pe \le 2$ (cell no larger than $2\alpha$). Below it, physical dispersion dominates and the plume is resolved; above it, **numerical** dispersion takes over and artificially smears the plume. With $\alpha_L = 10$ m and $\alpha_T = 1$ m on a ~50 m grid, the longitudinal direction is moderately under-resolved and the **transverse** direction is *badly* under-resolved.

In [ ]:
aL, aT = LOCKED_PARAMS['alh'], LOCKED_PARAMS['ath1']
print('Accuracy target : Pe <= 2  (cell size <= 2*alpha)')
print(f"  longitudinal  : max well-resolved cell = 2*alpha_L = {2*aL:.0f} m")
print(f"  transverse    : max well-resolved cell = 2*alpha_T = {2*aT:.0f} m")
print()
print(f"Native source-cell size dx = {dn['dx_src']:.0f} m")
print(f"  Pe_L = dx/alpha_L = {dn['dx_src']:.0f}/{aL:.0f} = {dn['PeL']:.1f}   "
      f"({'ABOVE' if dn['PeL'] > 2 else 'below'} the Pe<=2 target)")
print(f"  Pe_T = dx/alpha_T = {dn['dx_src']:.0f}/{aT:.0f} = {dn['PeT']:.0f}   "
      f"({'ABOVE' if dn['PeT'] > 2 else 'below'} the Pe<=2 target)")

### Checkpoint 1 — predict, then compute

First **predict** (no arithmetic): is the native grid above or below the $Pe_L \le 2$ accuracy threshold, and what does that imply for the plume? Then compute $Pe_L$ from the printed cell size.

In [ ]:
check_task_with_solution('task_t04_checkpoint_1')

---
## 6 — Refine the corridor and measure the recovery

Now we rebuild the **same doublet** on a copy of the grid where the source→receptor corridor is locally refined to **10 m cells** (`build_doublet_base` from the validated base-model module — it interpolates $K$, heads and the boundaries onto the refined mesh and re-solves the coupled flow + transport). Everything else is identical: same wells, same $Q$, same locked transport parameters.

We then overlay the two breakthrough curves and compare the recirculation the abstraction well sees.

In [ ]:
t0 = time.time()
ref_ucn = os.path.join(REFINED_WS, 'sim', 'gwt.ucn')
if (not FORCE_RERUN) and os.path.exists(ref_ucn):
    db = load_doublet_base(REFINED_WS); status_ref = 'cached'
else:
    if os.path.exists(REFINED_WS):
        shutil.rmtree(REFINED_WS)
    db = build_doublet_base(gwf_flow, boundary_gdf, rivers_gdf, INJ_XY, ABS_XY,
                            case_ws=REFINED_WS, source_mode='well_aux',
                            Q=Q_INJ, c_src=C_INJ, total_time=TOTAL_TIME)
    status_ref = 'built'
print(f'Refined doublet [{status_ref}] in {time.time() - t0:.0f} s')

dr = doublet_diagnostics(db.gwf, db.gwt, INJ_XY, ABS_XY)
print(f"REFINED doublet: {len(dr['xcc'])} cells, source-cell size dx = {dr['dx_src']:.0f} m")
print(f"  grid Peclet  : Pe_L = {dr['PeL']:.1f}   Pe_T = {dr['PeT']:.0f}")
print(f"  recirculation to abstraction well : {dr['recirc']:.3f} of c_inj")

In [ ]:
rec_gain  = dr['recirc']   / dn['recirc']   - 1
mid_gain  = dr['mid_peak'] / dn['mid_peak'] - 1
fmax_gain = dr['field_max']/ dn['field_max']- 1

fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1.6, 1]})

axA.plot(dn['times'], dn['bt_well'], color='firebrick', lw=1.6, ls='--',
         label=f"native, well  (plateau {dn['recirc']:.2f})")
axA.plot(dr['times'], dr['bt_well'], color='firebrick', lw=2.4,
         label=f"refined, well (plateau {dr['recirc']:.2f})")
axA.plot(dn['times'], dn['bt_mid'],  color='goldenrod', lw=1.4, ls='--',
         label=f"native, mid  (peak {dn['mid_peak']:.2f})")
axA.plot(dr['times'], dr['bt_mid'],  color='goldenrod', lw=2.0,
         label=f"refined, mid (peak {dr['mid_peak']:.2f})")
axA.set_xlabel('time (d)'); axA.set_ylabel('C / c_inj'); axA.set_ylim(0, 1)
axA.set_title('Breakthrough: native (dashed) vs refined (solid)')
axA.legend(fontsize=8); axA.grid(alpha=0.3)

labels = ['recirc.\n(well)', 'peak\n(mid)', 'plume\nmax']
nat_v  = [dn['recirc'], dn['mid_peak'], dn['field_max']]
ref_v  = [dr['recirc'], dr['mid_peak'], dr['field_max']]
x = np.arange(3); w = 0.38
axB.bar(x - w/2, nat_v, w, color='lightcoral', label='native')
axB.bar(x + w/2, ref_v, w, color='firebrick',  label='refined')
for xi, (nv, rv) in enumerate(zip(nat_v, ref_v)):
    axB.text(xi, max(nv, rv) + 0.02, f'+{100*(rv/nv-1):.0f}%', ha='center', fontsize=9)
axB.set_xticks(x); axB.set_xticklabels(labels); axB.set_ylim(0, 1.1)
axB.set_ylabel('C / c_inj'); axB.set_title('Coarse grid under-predicts'); axB.legend()
plt.tight_layout(); plt.show()

print(f"Refining the corridor recovers:")
print(f"  recirculation at the abstraction well : {dn['recirc']:.3f} -> {dr['recirc']:.3f}  (+{100*rec_gain:.0f}%)")
print(f"  peak concentration mid-corridor       : {dn['mid_peak']:.3f} -> {dr['mid_peak']:.3f}  (+{100*mid_gain:.0f}%)")
print(f"  plume maximum concentration           : {dn['field_max']:.3f} -> {dr['field_max']:.3f}  (+{100*fmax_gain:.0f}%)")

**The punchline.** On the native (~54 m) grid the **peak concentration** along the source→receptor corridor came out at ~0.84 of $c_{inj}$; refining the corridor to 10 m cells lifts it to ~0.98 — the coarse grid **under-predicted the peak centre-line concentration by ~16%**, purely because numerical dispersion spread the narrow plume sideways and diluted the centre-line the abstraction well draws from. Refining the corridor recovered most of that signal. The **flux-integrated recirculation fraction** at the well behaves the *same way* here — the coarse abstraction cell dilutes it too (~0.18 → ~0.30 on refining) — but, crucially, it **converges** to a stable value as the grid sharpens.

This is the decision-relevant lesson: **a coarse transport grid does not fail loudly; it quietly under-states concentrations along the corridor — both the peak *and* the recirculation the well sees — so the contamination *intensity* looks milder than it is.** If you read a peak concentration, a recirculation fraction or a contaminant arrival off an unrefined grid, you are biased toward under-estimating it. The asymmetry that matters for §9 is a different one: the recirculation fraction and the corridor peak **converge** as you refine (so a refined estimate is defensible), but the lateral plume width does **not** — no feasible grid resolves it.

---
## 7 — Transverse adequacy: what refinement *cannot* fix

Refining the corridor brought $Pe_L$ to about 1 — the longitudinal direction is now well resolved. But look at the **transverse** Péclet: $\alpha_T = 1$ m needs cells of ~2 m to resolve, and even the 10 m refined corridor leaves $Pe_T \approx 10\text{–}12 \gg 2$. To reach $Pe_T \le 2$ you would need ~2 m cells throughout the corridor — tens of thousands of cells, minute time steps, and still on a DISV mesh that cannot perfectly align with the flow. **Transverse spreading stays numerically over-estimated at any feasible resolution.**

The maps below make it concrete: refining sharpens and concentrates the plume, but its lateral *width* is governed by a dispersion the grid still cannot resolve.

In [ ]:
fig, (axN, axR) = plt.subplots(1, 2, figsize=(13, 5.2))
plot_plume(axN, dn, f"Native   (Pe_T = {dn['PeT']:.0f})")
caR = plot_plume(axR, dr, f"Refined  (Pe_T = {dr['PeT']:.0f})")
fig.colorbar(caR, ax=[axN, axR], shrink=0.7, label='C / c_inj')
plt.show()

print(f"Transverse grid Peclet (need <= 2 to resolve lateral spreading):")
print(f"  native  Pe_T = {dn['PeT']:.0f}")
print(f"  refined Pe_T = {dr['PeT']:.0f}   -> still >> 2: lateral width is NOT resolved")
print(f"  to reach Pe_T <= 2 would need cells <= {2*LOCKED_PARAMS['ath1']:.0f} m everywhere in the corridor")

### Checkpoint 3 — defensible threshold claims

Given that even the refined grid has $Pe_T \gg 2$, decide which reported result you could defend to a regulator.

In [ ]:
create_multiple_choice('task_t04_checkpoint_3')

---
## 8 — The Péclet–Courant coupling

Refining space is not free: it forces refining **time**. The Courant number $Cr = v\,\Delta t/\Delta s \le 1$ is the temporal-accuracy criterion — a solute parcel should not jump more than one cell per step, or the breakthrough curve smears. When the cell size $\Delta s$ shrinks, the admissible $\Delta t$ shrinks with it, and the number of time steps (hence run time) climbs. Below we read the binding (well-adjacent, non-sliver) cell from each grid's specific-discharge field and compute its maximum admissible step.

In [ ]:
print('Binding corridor cell (well-adjacent, sliver-floored):')
print(f"  native  : v = {dn['v_bind']:.2f} m/d,  ds = {dn['ds_bind']:.0f} m  "
      f"-> dt_max(Cr<=1) = {dn['dt_max']:.2f} d")
print(f"  refined : v = {dr['v_bind']:.2f} m/d,  ds = {dr['ds_bind']:.1f} m  "
      f"-> dt_max(Cr<=1) = {dr['dt_max']:.2f} d")
print()
print(f"Refining the corridor from ~{dn['ds_bind']:.0f} m to ~{dr['ds_bind']:.0f} m cells cut the")
print(f"admissible time step by ~{dn['dt_max']/dr['dt_max']:.0f}x  -> ~{dn['dt_max']/dr['dt_max']:.0f}x more time steps for the same simulation.")
print('Accuracy in space and accuracy in time are coupled: you pay for both.')

### Checkpoint 2 — maximum time step on the refined grid

Using the **refined** well-adjacent cell ($v \approx 28.3$ m/d, $\Delta s \approx 10.8$ m), compute the largest $\Delta t$ that keeps $Cr \le 1$.

In [ ]:
check_task_with_solution('task_t04_checkpoint_2')

---
## 9 — Summary and handoff

You built a coupled steady-flow / transient-transport GWHE doublet, diagnosed its grid, and confronted the central trade-off of transport modelling.

### What you're taking forward

| Result on this doublet | Native (~54 m) | Refined (10 m corridor) | Trust it? |
|---|---|---|---|
| Grid Péclet $Pe_L$ | ≈ 5.4 (above 2) | ≈ 1.1 | longitudinal: resolved only after refining |
| Grid Péclet $Pe_T$ | ≈ 54 | ≈ 11 (still ≫ 2) | transverse: **never** resolved at feasible cost |
| Peak concentration (corridor) | ≈ 0.84 | ≈ 0.98 (+16%) | under-predicted on coarse grid; refine before quoting |
| Recirculation to the well | ≈ 0.18 | ≈ 0.30 | under-read on coarse grid but **converges** on refinement — defensible once refined |
| Lateral plume width / contour | — | — | **not defensible** (numerical artefact) |
| Max time step ($Cr\le1$) | ≈ 6.6 d | ≈ 0.38 d | refining space forces refining time (~17×) |

**Key takeaways**
- $Pe \le 2$ and $Cr \le 1$ are **accuracy** criteria, not stability limits — TVD + implicit IMS stay stable above them; they smear rather than blow up.
- A coarse grid **under-predicts concentrations** along the corridor — both the peak *and* the recirculation the well sees — so it biases you toward thinking the contamination *intensity* is *milder* than it is.
- Local refinement **recovers** the longitudinal / flux-integrated quantities (peak, recirculation fraction) — they converge to defensible values — but **cannot** fix the transverse under-resolution, so exact lateral-width / contour claims are never defensible.

> **A note on progress markers.** The flow notebooks stamp section-completion markers; the transport track does not yet have a shared marker scaffold, so this notebook leaves them out (consistent with 01t–03t). A track-wide marker decision is a deferred item.

This grid-judgment carries directly into **Step 8 (Application)**, where the refined doublet is the keystone model for evaluating the heat-exchange scheme.

---

**Navigation:** [< Step 3: MODFLOW for Transport](03t_modflow_transport.ipynb) | [Step 5: Calibration >](05t_calibration.ipynb)

## References

\[1\] Langevin, C.D., Hughes, J.D., Provost, A.M., Banta, E.R., Niswonger, R.G., Panday, S. (2022): *MODFLOW 6 — Description of Input and Output.* U.S. Geological Survey Techniques and Methods 6-A62.

\[2\] Zheng, C., Bennett, G.D. (2002): *Applied Contaminant Transport Modeling*, 2nd ed. Wiley. (Grid Péclet and Courant accuracy criteria.)

\[3\] FloPy Documentation: Groundwater Transport (GWT). [https://flopy.readthedocs.io/](https://flopy.readthedocs.io/) (accessed 2026)